# Economía y Tienda — Plantastic Coins

Este notebook demuestra el sistema de economía del simulador:

- Consultar el saldo de **Plantastic Coins** de un usuario
- Listar los ítems disponibles en la **tienda**
- **Comprar** un ítem y ver cómo se descuenta el saldo
- Ver el **inventario** del usuario
- Probar la validación de **saldo insuficiente**
- Cuidar una planta y ver cómo se **ganan coins**

> **Requisito:** El backend debe estar corriendo en `http://127.0.0.1:8000`  
> Ejecuta `iniciar_backend_y_tunel.bat` antes de correr este notebook.

## 1. Configuración

In [1]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

# Cambia estos valores por un usuario y planta reales de tu DB
USER_ID  = 22
PLANT_ID = 9

def pretty(resp):
    """Imprime la respuesta JSON de forma legible."""
    print(f"HTTP {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    except Exception:
        print(resp.text)

print("Configuración lista ✓")

Configuración lista ✓


## 2. Consultar saldo de Plantastic Coins

In [2]:
resp = requests.get(f"{BASE_URL}/economy/users/{USER_ID}/wallet")
pretty(resp)

HTTP 200
{
  "user_id": 22,
  "coins_balance": 100
}


## 3. Listar todos los ítems de la tienda

In [3]:
resp = requests.get(f"{BASE_URL}/shop/items")
pretty(resp)

HTTP 200
{
  "total": 6,
  "items": [
    {
      "id_item": 5,
      "item_name": "Roca decorativa musgo",
      "item_type": "decoracion",
      "price_coins": 20,
      "rarity": "common",
      "active": true
    },
    {
      "id_item": 6,
      "item_name": "Luciérnagas mágicas",
      "item_type": "decoracion",
      "price_coins": 75,
      "rarity": "epic",
      "active": true
    },
    {
      "id_item": 3,
      "item_name": "Fondo Bosque de Niebla",
      "item_type": "fondo",
      "price_coins": 30,
      "rarity": "common",
      "active": true
    },
    {
      "id_item": 4,
      "item_name": "Fondo Atardecer Andino",
      "item_type": "fondo",
      "price_coins": 60,
      "rarity": "epic",
      "active": true
    },
    {
      "id_item": 1,
      "item_name": "Maceta de barro clásica",
      "item_type": "maceta",
      "price_coins": 25,
      "rarity": "common",
      "active": true
    },
    {
      "id_item": 2,
      "item_name": "Maceta minimalista bla

## 4. Filtrar ítems por tipo

Los tipos disponibles son: `maceta`, `fondo`, `decoracion`

In [4]:
for tipo in ["maceta", "fondo", "decoracion"]:
    print(f"\n{'='*40}")
    print(f"  Tipo: {tipo}")
    print(f"{'='*40}")
    resp = requests.get(f"{BASE_URL}/shop/items", params={"item_type": tipo})
    pretty(resp)


  Tipo: maceta
HTTP 200
{
  "total": 2,
  "items": [
    {
      "id_item": 1,
      "item_name": "Maceta de barro clásica",
      "item_type": "maceta",
      "price_coins": 25,
      "rarity": "common",
      "active": true
    },
    {
      "id_item": 2,
      "item_name": "Maceta minimalista blanca",
      "item_type": "maceta",
      "price_coins": 45,
      "rarity": "rare",
      "active": true
    }
  ]
}

  Tipo: fondo
HTTP 200
{
  "total": 2,
  "items": [
    {
      "id_item": 3,
      "item_name": "Fondo Bosque de Niebla",
      "item_type": "fondo",
      "price_coins": 30,
      "rarity": "common",
      "active": true
    },
    {
      "id_item": 4,
      "item_name": "Fondo Atardecer Andino",
      "item_type": "fondo",
      "price_coins": 60,
      "rarity": "epic",
      "active": true
    }
  ]
}

  Tipo: decoracion
HTTP 200
{
  "total": 2,
  "items": [
    {
      "id_item": 5,
      "item_name": "Roca decorativa musgo",
      "item_type": "decoracion",
      "p

## 5. Comprar un ítem

Compra el ítem con `item_id=1`. Ajusta según los IDs que retornó la celda anterior.

In [5]:
ITEM_ID = 1  # Ajusta al ID del ítem que quieres comprar

resp = requests.post(
    f"{BASE_URL}/shop/purchase",
    json={"user_id": USER_ID, "item_id": ITEM_ID, "quantity": 1}
)
pretty(resp)

HTTP 200
{
  "exito": true,
  "mensaje": "Compra realizada con éxito.",
  "item": {
    "id_item": 1,
    "item_name": "Maceta de barro clásica",
    "item_type": "maceta",
    "price_coins": 25
  },
  "quantity": 1,
  "costo": 25,
  "coins_balance": 75
}


## 6. Ver saldo después de la compra

In [6]:
resp = requests.get(f"{BASE_URL}/economy/users/{USER_ID}/wallet")
pretty(resp)

HTTP 200
{
  "user_id": 22,
  "coins_balance": 75
}


## 7. Ver inventario del usuario

In [7]:
resp = requests.get(f"{BASE_URL}/shop/users/{USER_ID}/inventory")
pretty(resp)

HTTP 200
{
  "total": 1,
  "inventario": [
    {
      "id_inventory": 1,
      "id_item": 1,
      "item_name": "Maceta de barro clásica",
      "item_type": "maceta",
      "rarity": "common",
      "quantity": 1,
      "acquired_at": "2026-04-24 13:58:28.036325"
    }
  ]
}


## 8. Probar saldo insuficiente

Intenta comprar el ítem más caro (Luciérnagas mágicas, 75 coins). Si el saldo es bajo, debería retornar HTTP 400.

In [8]:
ITEM_CARO_ID = 6  # Ajusta al ID del ítem más caro
CANTIDAD_EXAGERADA = 999

resp = requests.post(
    f"{BASE_URL}/shop/purchase",
    json={"user_id": USER_ID, "item_id": ITEM_CARO_ID, "quantity": CANTIDAD_EXAGERADA}
)
pretty(resp)

if resp.status_code == 400:
    print("\n✓ Validación de saldo insuficiente funcionando correctamente")

HTTP 400
{
  "exito": false,
  "mensaje": "Saldo insuficiente para completar la compra.",
  "coins_balance": 75,
  "costo": 74925
}

✓ Validación de saldo insuficiente funcionando correctamente


## 9. Ganar coins cuidando la planta

Cuando todos los factores están en rango óptimo y la salud ≥ 80, cada acción de cuidado otorga **1 Plantastic Coin**.

In [11]:
# Primero revisa el estado actual de la planta
resp = requests.get(f"{BASE_URL}/plants/{PLANT_ID}/status")
print(f"HTTP {resp.status_code}")

data = resp.json()
planta = data.get("planta") if isinstance(data, dict) else None

# Si no llega el objeto planta, mostramos la respuesta completa para depurar
if not planta:
    print("No se pudo leer el estado de la planta. Respuesta completa:")
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print("Estado actual:")
    print(f"  Agua       : {planta.get('water_level', '?')}")
    print(f"  Luz        : {planta.get('light_level', '?')}")
    print(f"  Humedad    : {planta.get('humidity_level', '?')}")
    print(f"  Ventilación: {planta.get('ventilation_level', '?')}")
    print(f"  Salud      : {planta.get('health', '?')}")

    alertas = data.get("alertas", []) if isinstance(data, dict) else []
    print(f"\nAlertas: {alertas}")

HTTP 200
Estado actual:
  Agua       : 29.037479246714287
  Luz        : 70.0
  Humedad    : 71.83151354466669
  Ventilación: 85.0
  Salud      : 64.86

Alertas: ['💧 Agua baja (29.0%): Marchitamiento de hojas, raíces en estrés hídrico.', '💨 Ventilación elevada (85.0%): Sustrato se seca demasiado rápido, raíces deshidratadas.']


In [ ]:
def revisar_condiciones_optimas(plant_id):
    status_resp = requests.get(f"{BASE_URL}/plants/{plant_id}/status")
    print(f"\n[STATUS] HTTP {status_resp.status_code}")

    data = status_resp.json()
    planta = data.get("planta") if isinstance(data, dict) else None

    if not planta:
        print("No se pudo leer la planta en /status:")
        print(json.dumps(data, indent=2, ensure_ascii=False))
        return None

    w = float(planta.get("water_level", 0))
    l = float(planta.get("light_level", 0))
    h = float(planta.get("humidity_level", 0))
    v = float(planta.get("ventilation_level", 0))
    health = float(planta.get("health", 0))

    print("Valores actuales:")
    print(f"  water_level      = {w}")
    print(f"  light_level      = {l}")
    print(f"  humidity_level   = {h}")
    print(f"  ventilation_level= {v}")
    print(f"  health           = {health}")

    checks = {
        "water 40-80": 40 <= w <= 80,
        "light 40-80": 40 <= l <= 80,
        "humidity 40-70": 40 <= h <= 70,
        "ventilation 30-70": 30 <= v <= 70,
        "health >= 80": health >= 80,
    }

    print("\nCondiciones para ganar coins:")
    for k, ok in checks.items():
        print(f"  {'OK' if ok else 'FAIL'} - {k}")

    return checks


def ejecutar_cuidado(label, url, payload):
    print(f"\n→ {label}...")
    r = requests.post(url, json=payload)
    print(f"  HTTP: {r.status_code}")
    data = r.json()
    print(f"  Respuesta: {json.dumps(data, ensure_ascii=False)}")
    print(f"  Coins ganadas: {data.get('coins_ganadas', 0)}")


wallet_before_resp = requests.get(f"{BASE_URL}/economy/users/{USER_ID}/wallet")
wallet_before = wallet_before_resp.json()
saldo_antes = wallet_before.get("coins_balance", 0)
print(f"Saldo antes de cuidar: {saldo_antes} coins")

# Diagnóstico inicial de condiciones
revisar_condiciones_optimas(PLANT_ID)

# Acciones de cuidado (payloads correctos según API)
ejecutar_cuidado(
    "Regando planta",
    f"{BASE_URL}/plants/{PLANT_ID}/water",
    {"cantidad_ml": 10},
)
ejecutar_cuidado(
    "Ajustando luz",
    f"{BASE_URL}/plants/{PLANT_ID}/light",
    {"intensidad": 50},
)
ejecutar_cuidado(
    "Ajustando ventilación",
    f"{BASE_URL}/plants/{PLANT_ID}/ventilation",
    {"nivel": 50},
)

# Diagnóstico final
revisar_condiciones_optimas(PLANT_ID)

wallet_after_resp = requests.get(f"{BASE_URL}/economy/users/{USER_ID}/wallet")
wallet_after = wallet_after_resp.json()
saldo_despues = wallet_after.get("coins_balance", 0)
print(f"\nSaldo después de cuidar: {saldo_despues} coins")
print(f"Diferencia: +{saldo_despues - saldo_antes} coins")

Saldo antes de cuidar: 75 coins

[STATUS] HTTP 200
Valores actuales:
  water_level      = 29.037479246714287
  light_level      = 70.0
  humidity_level   = 71.83151354466669
  ventilation_level= 85.0
  health           = 64.86

Condiciones para ganar coins:
  FAIL - water 40-80
  OK - light 40-80
  FAIL - humidity 40-70
  FAIL - ventilation 30-70
  FAIL - health >= 80

→ Regando planta...
  HTTP: 422
  Respuesta: {"detail": [{"type": "missing", "loc": ["body", "cantidad_ml"], "msg": "Field required", "input": {"user_id": 22, "amount": 10}}]}
  Coins ganadas: 0

→ Ajustando luz...
  HTTP: 422
  Respuesta: {"detail": [{"type": "missing", "loc": ["body", "intensidad"], "msg": "Field required", "input": {"user_id": 22, "amount": 5}}]}
  Coins ganadas: 0

→ Ajustando ventilación...
  HTTP: 422
  Respuesta: {"detail": [{"type": "missing", "loc": ["body", "nivel"], "msg": "Field required", "input": {"user_id": 22, "amount": 5}}]}
  Coins ganadas: 0

[STATUS] HTTP 200
Valores actuales:
  water